# 07 — Vector Search
## From cosine similarity to an approximate index

**Prerequisites:** NumPy arrays, matrix multiplication, and a rough idea of "search / retrieval".

### Learning goals

By the end of this notebook you should be able to:

- explain why embeddings turn "find related items" into "find nearby vectors";
- write cosine-similarity search with NumPy, and say why normalising first turns it into a plain dot product;
- read `np.argpartition` as a "partial sort";
- describe how an IVF index trades a bit of accuracy for speed, and step through the clustering it is built on; and
- design a benchmark that measures both retrieval quality and cost.

Every instructional example is complete and executable. Only the final project is intentionally unfinished.

### How to use this notebook

Run the cells from top to bottom. Every code cell is finished and runnable. Only the last cell (the project) is left for you.

You will see:

- **Predict** — before you run a cell, write down what you expect it to print, and why.
- **What you just saw** — a short note after a cell.
- **`assert` lines** — the specification.

## 1. Embeddings turn items into points in space

An **embedding** is a list of numbers that represents an item — a sentence, an image. Similar items get nearby vectors, so "find related items" becomes "find the closest vectors to this one".

**Cosine similarity** between vectors `a` and `b` measures whether they point the same way:

```
cos(a, b) = (a . b) / (|a| * |b|)
```

It runs from `-1` (opposite) through `0` (unrelated) to `1` (same direction). If you first scale every vector to length 1 (**normalise** it), the `|a| * |b|` part is just `1`, so cosine similarity becomes a plain dot product — and one matrix multiply scores the whole database at once.

(Euclidean distance is a *different* measure: it also reacts to how long the vectors are, not just their direction.)

**Predict.** For `query = [1, 0.1]` (normalised), which of the four database rows below get a **positive** score, and which get a negative one?

![Vector similarity](assets/vector_space.svg)

In [1]:
import numpy as np
from typing import Sequence

def normalize_rows(matrix: np.ndarray) -> np.ndarray:
    """Scale each row to length 1, so a dot product equals cosine similarity."""
    matrix = np.asarray(matrix, dtype=float)
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    if np.any(norms == 0):
        raise ValueError("a zero vector has no direction to compare")
    return matrix / norms

database = normalize_rows(np.array([[1, 0], [.8, .2], [0, 1], [-1, 0]], dtype=float))
query = normalize_rows(np.array([[1, .1]], dtype=float))[0]
scores = database @ query

for i, s in enumerate(scores):
    print(f"row {i} {database[i].round(2)} -> score {s:+.3f}")
print("rows 0 and 1 point roughly the query's way (positive); row 3 points opposite (negative)")

row 0 [1. 0.] -> score +0.995
row 1 [0.97 0.24] -> score +0.989
row 2 [0. 1.] -> score +0.100
row 3 [-1.  0.] -> score -0.995
rows 0 and 1 point roughly the query's way (positive); row 3 points opposite (negative)


### Cosine ignores length; normalising makes the dot product *be* the cosine

Two quick checks of the claims above.

**Predict** `cosine([1, 1], [10, 10])` and `cosine([1, 0], [0, 1])`.

In [2]:
def cosine(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

same_direction = np.array([[1., 1.], [10., 10.]])
orthogonal     = np.array([[1., 0.], [0., 1.]])

print("cos([1,1], [10,10]) =", round(cosine(*same_direction), 6), " <- same direction, length ignored")
print("Euclidean distance  =", round(float(np.linalg.norm(same_direction[0] - same_direction[1])), 2),
      " <- but they ARE far apart in distance")
print("cos([1,0], [0,1])   =", cosine(*orthogonal), " <- at right angles -> 0")

# After normalising, one matrix multiply gives every pairwise cosine.
unit = normalize_rows(np.array([[3., 4.], [1., 0.], [-2., 0.]]))
gram = unit @ unit.T
print("\nunit @ unit.T =\n", gram.round(3))
print("diagonal is 1 (each vector with itself); gram[1,2] = -1 ([1,0] vs [-2,0], opposite)")

assert np.allclose(np.diag(gram), 1.0)
assert np.isclose(gram[1, 2], cosine(unit[1], unit[2]))

cos([1,1], [10,10]) = 1.0  <- same direction, length ignored
Euclidean distance  = 12.73  <- but they ARE far apart in distance
cos([1,0], [0,1])   = 0.0  <- at right angles -> 0

unit @ unit.T =
 [[ 1.   0.6 -0.6]
 [ 0.6  1.  -1. ]
 [-0.6 -1.   1. ]]
diagonal is 1 (each vector with itself); gram[1,2] = -1 ([1,0] vs [-2,0], opposite)


## 2. Exact top-k search

Exact search scores the query against **every** database vector and returns the true `k` nearest. One matrix multiply does the scoring.

For the "pick the top `k`" step, a full sort is wasteful. `np.argpartition` does a **partial sort**: it only guarantees the `k` largest land in the last `k` slots (in *some* order). That is `O(n)` instead of `O(n log n)`. Then we sort just those `k` for a stable ranking.

**Predict.** `np.argpartition([30, 10, 20, 50, 40], -2)[-2:]` — which two indices, and are they in order?

In [3]:
scores_demo = np.array([30, 10, 20, 50, 40])

top2_unordered = np.argpartition(scores_demo, -2)[-2:]
print("argpartition top-2 (positions):", top2_unordered.tolist(),
      "  values:", scores_demo[top2_unordered].tolist(), " <- the two biggest, but not sorted")

ranked = top2_unordered[np.argsort(scores_demo[top2_unordered])[::-1]]
print("after sorting just those two :", ranked.tolist(),
      "  values:", scores_demo[ranked].tolist(), " <- 50 before 40")

assert set(top2_unordered.tolist()) == {3, 4}
assert ranked.tolist() == [3, 4]

argpartition top-2 (positions): [4, 3]   values: [40, 50]  <- the two biggest, but not sorted
after sorting just those two : [3, 4]   values: [50, 40]  <- 50 before 40


In [4]:
def exact_search(query: np.ndarray, database: np.ndarray, k: int = 5) -> tuple[np.ndarray, np.ndarray]:
    if not 1 <= k <= len(database):
        raise ValueError("k must be in [1, database size]")
    scores = database @ query
    candidates = np.argpartition(scores, -k)[-k:]                  # top k, unordered
    order = candidates[np.argsort(scores[candidates])[::-1]]      # sort just those k
    return order, scores[order]

indices, similarities = exact_search(query, database, k=2)
print("top-2 rows :", indices.tolist())
print("their scores:", similarities.round(3).tolist())
assert indices.tolist() == [0, 1]

top-2 rows : [0, 1]
their scores: [0.995, 0.989]


## 3. A smaller search: the IVF index

Scoring every vector is fine for a few thousand. For millions it is slow. An **IVF index** ("inverted file") speeds it up:

1. **Build time:** group the database vectors into clusters (with k-means). Each cluster has a centre (**centroid**) and a list of the vectors in it (a **posting list**).
2. **Query time:** compare the query to the few **centroids**, pick the nearest `probes` clusters, and run exact search on just those vectors.

This is **approximate**: if the true nearest neighbour sits in a cluster you did not probe, you miss it. That is measured by **recall** — the fraction of the true top-`k` you actually found. More `probes` → higher recall, more work.

The k-means below is the simple "assign, then re-centre, repeat" version. It is for learning, not for production.

One detail: `kmeans` **recomputes `labels` against the centroids it returns**. Otherwise an early stop could hand back centroids from one round and labels from the round before, and the posting lists would not match the centroids.

In [5]:
def kmeans(x: np.ndarray, clusters: int, iterations: int = 15, seed: int = 7):
    rng = np.random.default_rng(seed)
    centroids = x[rng.choice(len(x), clusters, replace=False)].copy()   # random starting centres
    for _ in range(iterations):
        labels = np.argmax(x @ normalize_rows(centroids).T, axis=1)     # assign each vector to nearest centre
        updated = np.vstack([x[labels == c].mean(axis=0) if np.any(labels == c) else centroids[c]
                             for c in range(clusters)])                 # re-centre
        if np.allclose(updated, centroids):
            break
        centroids = updated
    centroids = normalize_rows(centroids)
    labels = np.argmax(x @ centroids.T, axis=1)                         # labels match the RETURNED centroids
    return centroids, labels

class IVFIndex:
    def fit(self, vectors: np.ndarray, clusters: int = 8):
        self.vectors = normalize_rows(vectors)
        self.centroids, labels = kmeans(self.vectors, clusters)
        self.postings = [np.flatnonzero(labels == c) for c in range(clusters)]   # vectors per cluster
        return self

    def search(self, query: np.ndarray, k: int, probes: int = 1):
        query = normalize_rows(np.asarray(query)[None, :])[0]
        chosen = np.argsort(self.centroids @ query)[-probes:]                     # nearest `probes` centroids
        candidates = np.unique(np.concatenate([self.postings[c] for c in chosen]))
        if not len(candidates):
            return np.array([], dtype=int), np.array([])
        kk = min(k, len(candidates))
        local, scores = exact_search(query, self.vectors[candidates], kk)          # exact search on the shortlist
        return candidates[local], scores

print("kmeans and IVFIndex defined")

kmeans and IVFIndex defined


### Watch k-means run

The next cell makes three obvious blobs of points, runs `kmeans`, and prints where each centroid ended up and how many points it captured.

**Predict** roughly which direction each centroid points, and whether the three clusters come out the same size.

In [6]:
rng2 = np.random.default_rng(0)
blobs = normalize_rows(np.vstack([
    rng2.normal(centre, 0.05, size=(20, 2))
    for centre in ([1.0, 0.0], [0.0, 1.0], [-1.0, -1.0])
]))
centroids3, labels3 = kmeans(blobs, clusters=3, seed=0)

for c in range(3):
    print(f"cluster {c}: {int(np.sum(labels3 == c)):2d} points, centroid points ≈ {centroids3[c].round(2)}")

# The labels kmeans returned match a fresh assignment to the centroids it returned.
assert np.array_equal(labels3, np.argmax(blobs @ centroids3.T, axis=1))
assert sorted(np.bincount(labels3).tolist()) == [20, 20, 20]

# k-means depends on where the centres start. IVFIndex.fit() uses a different (default) seed,
# which on this same data splits it unevenly -- one cluster grabs most of the points.
posting_sizes = sorted(len(p) for p in IVFIndex().fit(blobs, 3).postings)
print("\nIVFIndex cluster sizes (its default seed):", posting_sizes, " <- lopsided, but still a valid split")
assert sum(posting_sizes) == 60

cluster 0: 20 points, centroid points ≈ [1.   0.01]
cluster 1: 20 points, centroid points ≈ [0.02 1.  ]
cluster 2: 20 points, centroid points ≈ [-0.71 -0.7 ]

IVFIndex cluster sizes (its default seed): [5, 15, 40]  <- lopsided, but still a valid split


### What you just saw

With a good starting point (`seed=0`) k-means recovered the three blobs exactly, 20 points each. With a different start (`IVFIndex`'s default) it landed in a worse split — one cluster swallowed most points. Both are *correct* partitions (every point is in exactly one cluster); the lopsided one just makes probing less useful, because probing one big cluster still checks most of the database.

That is why real systems run k-means several times and keep the best, or use a smarter initialisation.

### What `probes` buys you

Now on realistic data: 500 vectors, 32 dimensions, no built-in clusters. The next cell picks one query and increases `probes` from 1 to 12 (= the number of clusters), showing how many vectors get examined and what recall@10 comes out.

**Predict.** At `probes = 1`, is recall@10 closer to `0.1` or to `1.0`?

In [7]:
import time

rng = np.random.default_rng(7)
vectors = normalize_rows(rng.normal(size=(500, 32)))
index = IVFIndex().fit(vectors, clusters=12)
q = vectors[42]
exact_ids, _ = exact_search(q, vectors, 10)          # the true top 10

def recall_at_k(found: Sequence[int], expected: Sequence[int]) -> float:
    if not expected:
        return 1.0
    return len(set(found) & set(expected)) / len(set(expected))

q_unit = normalize_rows(q[None, :])[0]
print(f"{'probes':>6} {'vectors checked':>16} {'recall@10':>10}")
for probes in (1, 3, 6, 12):
    chosen = np.argsort(index.centroids @ q_unit)[-probes:]
    n_cand = len(np.unique(np.concatenate([index.postings[c] for c in chosen])))
    found, _ = index.search(q, 10, probes)
    print(f"{probes:>6} {n_cand:>16} {recall_at_k(found.tolist(), exact_ids.tolist()):>10.2f}")

assert recall_at_k(index.search(q, 10, 12)[0].tolist(), exact_ids.tolist()) == 1.0   # probing all = exact
print("\nprobes = 12 checks every vector, so recall is back to 1.0 -- that is just exact search again")

probes  vectors checked  recall@10
     1               46       0.40
     3              125       0.80
     6              249       0.80
    12              500       1.00

probes = 12 checks every vector, so recall is back to 1.0 -- that is just exact search again


### The cost side: latency

Higher recall is not free. The next cell adds timing (median and 95th-percentile microseconds) next to recall, reusing the same index and query.

**Predict.** At `probes = 12`, recall is `1.0`. Is the IVF index faster or slower than plain exact search here, on only 500 vectors?

In [8]:
def p50_p95_us(fn, repeats=300):
    ts = []
    for _ in range(repeats):
        s = time.perf_counter(); fn(); ts.append((time.perf_counter() - s) * 1e6)
    ts.sort()
    return ts[len(ts) // 2], ts[int(len(ts) * 0.95)]

e50, e95 = p50_p95_us(lambda: exact_search(q, vectors, 10))
print(f"{'probes':>6} {'recall@10':>10} {'p50 us':>9} {'p95 us':>9}")
print(f"{'exact':>6} {1.00:>10.2f} {e50:>9.1f} {e95:>9.1f}")
for probes in (1, 3, 6, 12):
    approximate, _ = index.search(q, 10, probes)
    r = recall_at_k(approximate.tolist(), exact_ids.tolist())
    a50, a95 = p50_p95_us(lambda p=probes: index.search(q, 10, p))
    print(f"{probes:>6} {r:>10.2f} {a50:>9.1f} {a95:>9.1f}")
print("\nOn 500 vectors, exact search is already tiny and the IVF bookkeeping only adds cost.")
print("IVF pays off at much larger scale -- run the project on a big corpus to see it flip.")

probes  recall@10    p50 us    p95 us
 exact       1.00      40.6      58.7
     1       0.40      73.9      81.5
     3       0.80      81.7      88.0
     6       0.80      98.1     103.4
    12       1.00     119.8     132.1

On 500 vectors, exact search is already tiny and the IVF bookkeeping only adds cost.
IVF pays off at much larger scale -- run the project on a big corpus to see it flip.


## 4. Judging an index on more than recall

Recall@k tells you how many true neighbours you found:

```
recall@k = (number of true top-k you returned) / k
```

But a real decision also needs: p50 / p95 latency, throughput, how long the index takes to build, how much memory it uses, and how it behaves with filters. And test with **held-out** queries — reusing database vectors as queries is too easy and makes the numbers look better than they are.

## Project — Semantic retrieval benchmark

Build an exact index and an IVF index for a labeled synthetic corpus. Create a held-out query set, sweep cluster counts and probe counts, and plot recall@10 against latency.

### Suggested milestones

1. Generate and normalize labeled clusters, then split database items from queries deterministically.
2. Implement a benchmark function that records exact and IVF results, latency, and index-build cost.
3. Compare several `(clusters, probes)` configurations against exact-search ground truth.
4. Plot the recall-latency frontier and inspect examples where the approximate index misses a neighbor.

You may `from course_utils import normalize_rows, exact_search, kmeans, IVFIndex, recall_at_k` instead of copying the cells above; the module ships the same implementations with docstrings.

**Acceptance criteria:** deterministic data split; no query leakage into tuning; p50 and p95 latency; a documented memory estimate; qualitative inspection of errors; and a recommendation justified by measurements. State the hardware, vector dimension, and timing method so results are reproducible.

**Checks to run yourself**

- Assert no query vector is byte-identical to any database vector after the split.
- For `probes == clusters`, assert IVF recall@k equals 1.0 (it probes everything).
- Increase the corpus to ~50k vectors and confirm IVF p95 latency drops below exact.
- Estimate index memory as `vectors.nbytes + centroids.nbytes + sum(p.nbytes for p in postings)` and check it against `tracemalloc`.
- Pull three queries where recall < 1.0 and look at which cluster the missed neighbor landed in.

In [ ]:
# PROJECT WORKSPACE — intentionally incomplete
#
# Reuse the retrieval code from this notebook, or import it:
#     from course_utils import normalize_rows, exact_search, kmeans, IVFIndex, recall_at_k
#
# 1. Generate labeled clusters; split database vs held-out queries deterministically
#    (fixed rng, no query vector left in the database).
# 2. benchmark_retrieval(corpus, queries, k): for each (clusters, probes) config record
#    recall@k vs exact ground truth, p50/p95 latency, and index build time.
# 3. Sweep several (clusters, probes); choose a recommendation from the measurements.
# 4. Plot recall@10 against p95 latency; inspect a few queries the IVF index gets wrong.

def benchmark_retrieval(corpus, queries, k=10):
    """Return recall / latency / build-cost results for exact and IVF retrieval."""
    raise NotImplementedError("Implement the retrieval benchmark")
